# TRELLIS.2：圖片 → 帶紋理 GLB（Colab）

由上往下依序執行即可，全新工作階段可完整重跑。

**事前準備**
- 執行階段選 **L4 或 A100**（需 24GB 以上顯存）。
- 左側「🔑 密鑰」新增 `HF_TOKEN`，並打開此筆記本的存取權。
- 輸入圖放在 Drive，路徑在第 1 格設定。

**重啟規則**：若第 5 格提示 transformers 不是 4.x，請「執行階段 → 重新啟動工作階段」（不要刪除），再從第 4 格（掛載 Drive）往下跑；前面的安裝與補丁都留在磁碟上。

跑完後記得「中斷並刪除執行階段」，或把第 1 格的 `AUTO_DISCONNECT` 設為 `True`。

In [ ]:
# ── 設定 ──────────────────────────────────────────
INPUT_DIR = "/content/drive/MyDrive/tmp/tank"    # 輸入圖資料夾（Drive）
OUT_DIR = "/content/drive/MyDrive/tmp/tank"      # GLB 輸出資料夾（Drive）
IMAGE_NAMES = ["snailer_tank.jpg"]               # 要處理的檔名；留空 [] = 處理資料夾內所有圖片
OVERWRITE = False                                # False：已有同名 GLB 就跳過

TEXTURE_SIZE = 4096        # 貼圖解析度；顯存吃緊或要輕量可用 2048
DECIMATION_TARGET = 1000000  # 目標面數；進遊戲建議 100000~200000

DOWNLOAD = False           # True：跑完把 GLB 下載到本機
AUTO_DISCONNECT = False    # True：全部跑完自動釋放執行階段（停止計費）

In [ ]:
%%bash
cd /content
[ -d TRELLIS.2 ] || git clone -b main https://github.com/microsoft/TRELLIS.2.git --recursive
cd TRELLIS.2 || exit 1
# 不裝 --flash-attn：Colab 目前的 PyTorch 2.11 / CUDA 13 / Python 3.13 組合編譯不起來，改走下面的 SDPA 補丁
. ./setup.sh --basic --nvdiffrast --nvdiffrec --cumesh --o-voxel --flexgemm
# RMBG 鏡像的自訂程式碼需要 transformers 4.x（DINOv3 需要 >= 4.56）
pip install -q "transformers>=4.56,<5"

In [ ]:
# 稀疏注意力補丁：把 xformers 分支改成逐序列 torch SDPA（不需要 flash-attn / xformers）
# 必須在 import trellis2 之前執行；可重複執行
import re

path = "/content/TRELLIS.2/trellis2/modules/sparse/attention/full_attn.py"
src = open(path).read()

pattern = re.compile(
    r"    if config\.ATTN == 'xformers':\n.*?(?=    elif config\.ATTN == 'flash_attn':\n)",
    re.S,
)

new_branch = """    if config.ATTN == 'xformers':
        # patched: per-sequence torch SDPA (no flash-attn / xformers CUDA needed)
        if num_all_args == 1:
            q, k, v = qkv.unbind(dim=1)
        elif num_all_args == 2:
            k, v = kv.unbind(dim=1)
        outs = []
        qi = ki = 0
        for lq, lk in zip(q_seqlen, kv_seqlen):
            qs = q[qi:qi + lq].transpose(0, 1).unsqueeze(0)    # [1, H, Lq, Ci]
            ks = k[ki:ki + lk].transpose(0, 1).unsqueeze(0)    # [1, H, Lk, Ci]
            vs = v[ki:ki + lk].transpose(0, 1).unsqueeze(0)    # [1, H, Lk, Co]
            o = torch.nn.functional.scaled_dot_product_attention(qs, ks, vs)
            outs.append(o[0].transpose(0, 1))                   # [Lq, H, Co]
            qi += lq
            ki += lk
        out = torch.cat(outs, dim=0)
"""

assert len(pattern.findall(src)) == 1, "找不到唯一的 xformers 分支，停止"
open(path, "w").write(pattern.sub(lambda m: new_branch, src))
print("patched")

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
from PIL import Image

IMG_EXT = {".jpg", ".jpeg", ".png", ".webp"}
in_dir, out_dir = Path(INPUT_DIR), Path(OUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)

if IMAGE_NAMES:
    image_paths = [in_dir / n for n in IMAGE_NAMES]
else:
    image_paths = sorted(p for p in in_dir.iterdir() if p.suffix.lower() in IMG_EXT)

missing = [p for p in image_paths if not p.exists()]
assert not missing, f"找不到輸入圖：{missing}"
print("待處理：", [p.name for p in image_paths])
Image.open(image_paths[0])

In [ ]:
import os, sys, subprocess
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["HF_HOME"] = "/content/drive/MyDrive/hf_cache"   # 模型快取放 Drive，下次不用重下載
os.environ["ATTN_BACKEND"] = "sdpa"
os.chdir("/content/TRELLIS.2")
sys.path.insert(0, "/content/TRELLIS.2")

import transformers
assert transformers.__version__.startswith("4."), (
    f"transformers 是 {transformers.__version__}，需要 4.x：請重新啟動工作階段後從第 4 格往下跑")

# gated 模型改用鏡像（全新 clone 後必須重做）；權重來源非官方，授權仍以原作者條款為準
PAIRS = [
    ("facebook/dinov3-vitl16-pretrain-lvd1689m", "camenduru/dinov3-vitl16-pretrain-lvd1689m"),
    ("briaai/RMBG-2.0", "1038lab/RMBG-2.0"),
]

from huggingface_hub import list_repo_files
for _, new in PAIRS:
    files = list_repo_files(new)
    assert "config.json" in files, f"{new} 缺 config.json，停止"
    assert any(f.endswith((".safetensors", ".bin", ".pth")) for f in files), f"{new} 沒有權重檔，停止"

for old, new in PAIRS:
    subprocess.run(
        f"grep -rl '{old}' /content/TRELLIS.2 /root/.cache/huggingface /content/drive/MyDrive/hf_cache 2>/dev/null"
        f" | xargs -r sed -i 's#{old}#{new}#g'",
        shell=True)

# BiRefNet 沒有 embedding / encoder-decoder 權重需要綁定；跳過以避開 get_text_config 錯誤
from transformers import modeling_utils
modeling_utils.PreTrainedModel.tie_embeddings_and_encoder_decoder = lambda self: None

from trellis2.pipelines import Trellis2ImageTo3DPipeline
import trellis2.modules.attention.config as attn_cfg
import trellis2.modules.sparse.config as sp_cfg
import o_voxel

attn_cfg.BACKEND = "sdpa"
sp_cfg.ATTN = "xformers"   # 實際走補丁後的 torch SDPA 分支
print("attention:", attn_cfg.BACKEND, "| sparse:", sp_cfg.ATTN)

pipeline = Trellis2ImageTo3DPipeline.from_pretrained("microsoft/TRELLIS.2-4B")
pipeline.cuda()

In [ ]:
import gc, time, torch

results = []
for p in image_paths:
    out = out_dir / f"{p.stem}.glb"
    if out.exists() and not OVERWRITE:
        print(f"skip（已存在）: {out.name}")
        results.append(out)
        continue

    t0 = time.time()
    mesh = pipeline.run(Image.open(p))[0]
    mesh.simplify(16777216)      # nvdiffrast 上限

    glb = o_voxel.postprocess.to_glb(
        vertices=mesh.vertices,
        faces=mesh.faces,
        attr_volume=mesh.attrs,
        coords=mesh.coords,
        attr_layout=mesh.layout,
        voxel_size=mesh.voxel_size,
        aabb=[[-0.5, -0.5, -0.5], [0.5, 0.5, 0.5]],
        decimation_target=DECIMATION_TARGET,
        texture_size=TEXTURE_SIZE,
        remesh=True,
        remesh_band=1,
        remesh_project=0,
        verbose=False,
    )
    glb.export(str(out), extension_webp=True)
    results.append(out)
    print(f"saved: {out}（{out.stat().st_size / 1e6:.1f} MB，{time.time() - t0:.0f}s）")

    del mesh, glb
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
if DOWNLOAD:
    from google.colab import files
    for out in results:
        files.download(str(out))

if AUTO_DISCONNECT:
    from google.colab import runtime
    runtime.unassign()   # 釋放執行階段，停止消耗運算單元